# V6.2 Sector Weight Optimization (GPU-Accelerated)

This notebook runs the full V6.2 weight optimization pipeline with **GPU acceleration**:

### Acceleration Architecture
1. **GPU Pre-computation** — All weight-independent indicators (HMA, StochRSI, ATR, RVOL) are
   computed ONCE for every (stock, 5-min bar) pair using CuPy on the T4 GPU. This eliminates
   ~99.9% of redundant indicator calculation across trials.
2. **Indicator Cache** — Pre-computed values are stored in a persistent dictionary. Each subsequent
   backtest run uses O(1) lookups instead of recomputing expensive indicators.
3. **Vectorised ATR** — Rolling ATR series for all stocks computed via GPU-parallelised cumsum.

### Expected Speedup
| Configuration | Without GPU | With GPU Precompute |
|:---|:---|:---|
| Smoke test (16 trials) | ~5 min | ~1-2 min |
| Medium run (432 trials) | ~2-4 hours | ~15-30 min |
| Full run (1264 trials) | ~6-12 hours | ~30-90 min |

### Pipeline Steps
1. **Discrete sanity pass** — zero-inclusive grid search
2. **Coarse LHS exploration** — Latin Hypercube space-filling
3. **Optuna TPE** — Bayesian single-objective optimization
4. **Optuna NSGA-II** — multi-objective Pareto refinement
5. **Nested walk-forward** — institutional-standard validation
6. **Deployment ranking** — baseline hurdles + fold-majority checks

**Parameters optimized:**
```
composite = structural_rs * W_structural
          + shortterm_rs  * W_shortterm
          + intraday_rs   * W_intraday
          + breadth       * W_breadth
          + nifty_pct     * W_nifty
```
Plus bias threshold `T` for LONG/SHORT/NEUTRAL gate.

## 1. Setup (GPU Runtime Required)

**IMPORTANT:** Before running, go to **Runtime → Change runtime type → T4 GPU**.
This enables CuPy-based GPU acceleration for indicator pre-computation.

In [ ]:
# Install required packages + GPU acceleration
!pip install -q optuna>=3.6.0 pyarrow>=14.0.0 scipy>=1.10.0

# Install CuPy for GPU (matches Colab's CUDA 12.x)
!pip install -q cupy-cuda12x>=13.0.0

# Verify GPU availability
import subprocess
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=5)
    gpu_info = result.stdout.strip()
    print(f"GPU detected: {gpu_info}")
except Exception:
    gpu_info = None
    print("WARNING: No GPU detected. Pre-computation will use NumPy (CPU) fallback.")

try:
    import cupy as cp
    print(f"CuPy version: {cp.__version__}")
    print(f"CuPy CUDA: {cp.cuda.runtime.runtimeGetVersion()}")
    x = cp.array([1, 2, 3])
    print(f"CuPy GPU test: OK ({cp.cuda.Device().name.decode()})")
except Exception as e:
    print(f"CuPy not available: {e}")
    print("Falling back to CPU-only mode (still uses indicator caching for speedup).")

In [ ]:
# Mount Google Drive (for data access)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# ── CONFIGURE THESE PATHS ──
# Path to this optimization package (where you uploaded it)
PACKAGE_DIR = "/content/v6_2_weight_opt_colab_package"

# Path to your parquet data (must contain daily/ and 5minute/ subfolders)
# Option A: From Google Drive
DATA_ROOT = "/content/drive/MyDrive/Python-Algo/backtest_v6/data"
# Option B: If you uploaded data alongside the package
# DATA_ROOT = f"{PACKAGE_DIR}/data"

# Optional: path to V6.2 analysis JSON for baseline hurdles
# Set to None to use hardcoded defaults
BASELINE_JSON = None
# BASELINE_JSON = "/content/drive/MyDrive/Python-Algo/backtest_v6/analysis_v6.2/advanced_analysis_24-02-2026_03-34_pm.json"

# Verify paths exist
assert Path(PACKAGE_DIR).exists(), f"Package not found at {PACKAGE_DIR}"
assert Path(DATA_ROOT).exists(), f"Data not found at {DATA_ROOT}"
assert (Path(DATA_ROOT) / "daily").exists(), f"Missing daily/ folder in {DATA_ROOT}"
assert (Path(DATA_ROOT) / "5minute").exists(), f"Missing 5minute/ folder in {DATA_ROOT}"

daily_count = len(list((Path(DATA_ROOT) / "daily").glob("*.parquet")))
intra_count = len(list((Path(DATA_ROOT) / "5minute").glob("*.parquet")))
print(f"Data root: {DATA_ROOT}")
print(f"  daily/*.parquet : {daily_count} files")
print(f"  5minute/*.parquet : {intra_count} files")
print("Setup OK.")

## 2. Smoke Test (Quick Validation)

Run a tiny optimization to verify everything works (including GPU pre-computation).
This uses minimal trials and a short date range. Should complete in 1-2 minutes.

In [ ]:
%%time
# Quick smoke test with GPU pre-computation: ~1-2 minutes
!python {PACKAGE_DIR}/run_weight_optimization.py \
  --data-root {DATA_ROOT} \
  --start-date 2025-03-01 \
  --end-date 2025-05-31 \
  --train-months 1 \
  --test-months 1 \
  --min-folds 1 \
  --discrete-trials 4 \
  --coarse-trials 4 \
  --tpe-trials 4 \
  --nsga-trials 4 \
  --shortlist-from-each 2 \
  --shortlist-size 4 \
  --min-trades 1 \
  --min-total-trades 1 \
  --results-dir {PACKAGE_DIR}/results_smoke

In [ ]:
# Verify smoke test output
import json
from pathlib import Path

smoke_dirs = sorted(Path(f"{PACKAGE_DIR}/results_smoke").glob("weight_opt_*"))
if smoke_dirs:
    latest = smoke_dirs[-1]
    print(f"Smoke results: {latest}")
    for f in sorted(latest.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
    report = json.loads((latest / "best_candidate_report.json").read_text())
    print(f"\nBest params: {report['selected_params']}")
    row = report['selected_row']
    print(f"OOS Return: {row.get('oos_return_pct', 0):.2f}%")
    print(f"OOS DD: {row.get('oos_drawdown_pct', 0):.2f}%")
    print(f"OOS PF: {row.get('oos_pf', 0):.3f}")
    print(f"Eligible: {row.get('eligible_for_deploy', False)}")
else:
    print("No smoke results found. Check output above for errors.")

## 3. Full Optimization Run

### Option A: Medium Run (~15-30 min with GPU, was 2-4 hours)
Good balance of exploration vs. compute time. GPU pre-computation makes the first few minutes
slightly slower (building indicator cache), then all 432 trials run at 10-50x speed.

In [ ]:
%%time
# Medium run with GPU acceleration: ~15-30 min (was 2-4 hours)
baseline_flag = f"--baseline-analysis-json {BASELINE_JSON}" if BASELINE_JSON else ""

!python {PACKAGE_DIR}/run_weight_optimization.py \
  --data-root {DATA_ROOT} \
  --start-date 2025-03-01 \
  --end-date 2025-12-31 \
  --train-months 4 \
  --test-months 1 \
  --min-folds 5 \
  --discrete-trials 32 \
  --coarse-trials 80 \
  --tpe-trials 200 \
  --nsga-trials 120 \
  --shortlist-from-each 20 \
  --shortlist-size 30 \
  --results-dir {PACKAGE_DIR}/results \
  {baseline_flag}

### Option B: Full Institutional Run (~30-90 min with GPU, was 6-12 hours)
Maximum exploration. Pre-computed indicator cache is amortised across all 1264 trials.

In [ ]:
%%time
# Full institutional run with GPU: ~30-90 min (was 6-12 hours)
baseline_flag = f"--baseline-analysis-json {BASELINE_JSON}" if BASELINE_JSON else ""

!python {PACKAGE_DIR}/run_weight_optimization.py \
  --data-root {DATA_ROOT} \
  --start-date 2025-03-01 \
  --end-date 2025-12-31 \
  --train-months 4 \
  --test-months 1 \
  --min-folds 5 \
  --discrete-trials 64 \
  --coarse-trials 200 \
  --tpe-trials 600 \
  --nsga-trials 400 \
  --shortlist-from-each 30 \
  --shortlist-size 40 \
  --results-dir {PACKAGE_DIR}/results \
  {baseline_flag}

In [ ]:
# import shutil
# import json
# from pathlib import Path
# from google.colab import files

# # Find the latest results folder (most recent timestamp)
# results_dirs = sorted(Path(f"{PACKAGE_DIR}/results").glob("weight_opt_*"))
# if results_dirs:
#     RESULT_DIR = results_dirs[-1]
#     print(f"✓ Found results: {RESULT_DIR.name}")
#     print(f"✓ Total files: {len(list(RESULT_DIR.iterdir()))}")
    
#     # Load and show quick summary
#     if (RESULT_DIR / "best_candidate_report.json").exists():
#         report = json.loads((RESULT_DIR / "best_candidate_report.json").read_text())
#         row = report["selected_row"]
#         print(f"\n🎯 Top Candidate:")
#         print(f"  Return: {row.get('oos_return_pct', 0):+.2f}% | DD: {row.get('oos_drawdown_pct', 0):.2f}% | PF: {row.get('oos_pf', 0):.3f}")
#         print(f"  Eligible: {row.get('eligible_for_deploy', False)}")
    
#     # Create zip and download
#     print(f"\n⏳ Creating zip archive...")
#     zip_filename = f"v6_2_opt_{RESULT_DIR.name}"
#     zip_path = Path(f"/tmp/{zip_filename}.zip")
#     shutil.make_archive(str(zip_path).replace('.zip', ''), 'zip', RESULT_DIR.parent, RESULT_DIR.name)
    
#     size_mb = zip_path.stat().st_size / (1024*1024)
#     print(f"✓ Zip created: {size_mb:.1f} MB")
#     print(f"\n📥 Downloading {zip_filename}.zip...")
#     files.download(str(zip_path))
#     print("✓ Download complete!")
# else:
#     print("❌ No results folder found. Run optimization first.")

## 4. Analyze Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Find latest results directory
results_dirs = sorted(Path(f"{PACKAGE_DIR}/results").glob("weight_opt_*"))
if not results_dirs:
    results_dirs = sorted(Path(f"{PACKAGE_DIR}/results_smoke").glob("weight_opt_*"))

RESULT_DIR = results_dirs[-1] if results_dirs else None
print(f"Results directory: {RESULT_DIR}")

if RESULT_DIR:
    print("\nOutput files:")
    for f in sorted(RESULT_DIR.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")

In [ ]:
# Load and display deployment ranking
if RESULT_DIR and (RESULT_DIR / "deployment_ranking.csv").exists():
    rank_df = pd.read_csv(RESULT_DIR / "deployment_ranking.csv")
    display_cols = [
        "phase", "w_structural", "w_shortterm", "w_intraday", "w_breadth",
        "w_nifty", "bias_threshold", "objective_score", "sparsity_near_zero",
        "oos_return_pct", "oos_drawdown_pct", "oos_pf", "oos_expectancy",
        "oos_trades", "overfit_gap", "eligible_for_deploy",
        "beats_default_oos_folds", "total_oos_folds"
    ]
    available_cols = [c for c in display_cols if c in rank_df.columns]
    print(f"Deployment Ranking ({len(rank_df)} candidates):")
    display(rank_df[available_cols].head(20))
else:
    print("No deployment_ranking.csv found.")

In [ ]:
# Load best candidate report
if RESULT_DIR and (RESULT_DIR / "best_candidate_report.json").exists():
    report = json.loads((RESULT_DIR / "best_candidate_report.json").read_text())

    print("=" * 70)
    print("BEST CANDIDATE REPORT")
    print("=" * 70)

    print("\nSelected Parameters:")
    for k, v in report["selected_params"].items():
        print(f"  {k:>15s} = {v}")

    row = report["selected_row"]
    print(f"\nOut-of-Sample Performance:")
    print(f"  Return:      {row.get('oos_return_pct', 0):+.2f}%")
    print(f"  Drawdown:    {row.get('oos_drawdown_pct', 0):.2f}%")
    print(f"  Profit Factor: {row.get('oos_pf', 0):.3f}")
    print(f"  Expectancy:  {row.get('oos_expectancy', 0):+.2f}")
    print(f"  Win Rate:    {row.get('oos_win_rate_pct', 0):.2f}%")
    print(f"  Trades:      {row.get('oos_trades', 0)}")
    print(f"  Overfit Gap: {row.get('overfit_gap', 0):.4f}")
    print(f"  Eligible:    {row.get('eligible_for_deploy', False)}")

    print(f"\nBaseline Hurdles:")
    for k, v in report.get("baseline_hurdles", {}).items():
        print(f"  {k:>20s} = {v}")

    print(f"\nBaseline OOS Aggregate:")
    base_agg = report.get("baseline_oos_aggregate", {})
    print(f"  Return:      {base_agg.get('return_pct', 0):+.2f}%")
    print(f"  Drawdown:    {base_agg.get('drawdown_pct', 0):.2f}%")
    print(f"  Profit Factor: {base_agg.get('profit_factor', 0):.3f}")
    print(f"  Trades:      {base_agg.get('trades', 0)}")

    stats = report.get("statistical_checks", {})
    if stats:
        print(f"\nStatistical Checks:")
        ci_ret = stats.get("bootstrap_return_ci95", {})
        print(f"  Return 95% CI:     [{ci_ret.get('low', 0):.2f}%, {ci_ret.get('high', 0):.2f}%]")
        ci_dd = stats.get("bootstrap_drawdown_ci95", {})
        print(f"  Drawdown 95% CI:   [{ci_dd.get('low', 0):.2f}%, {ci_dd.get('high', 0):.2f}%]")
        print(f"  Paired Fold Return Diff: {stats.get('paired_fold_mean_return_diff_pct', 0):+.2f}%")
        print(f"  Paired Fold DD Improvement: {stats.get('paired_fold_mean_drawdown_improvement_pct', 0):+.2f}%")

    print("=" * 70)
else:
    print("No best_candidate_report.json found.")

In [ ]:
# Load nested walk-forward results
if RESULT_DIR and (RESULT_DIR / "nested_walk_forward.json").exists():
    nested = json.loads((RESULT_DIR / "nested_walk_forward.json").read_text())

    print("Nested Walk-Forward Results")
    print("-" * 50)

    freq = nested.get("selected_config_frequency", {})
    print(f"\nConfig selection frequency (inner-loop winners):")
    for cfg_key, count in sorted(freq.items(), key=lambda x: -x[1]):
        print(f"  {count}x : {cfg_key}")

    agg = nested.get("aggregated_oos_metrics", {})
    print(f"\nAggregated OOS metrics (nested):")
    print(f"  Return:       {agg.get('return_pct', 0):+.2f}%")
    print(f"  Drawdown:     {agg.get('drawdown_pct', 0):.2f}%")
    print(f"  Profit Factor:{agg.get('profit_factor', 0):.3f}")
    print(f"  Expectancy:   {agg.get('expectancy', 0):+.2f}")
    print(f"  Trades:       {agg.get('trades', 0)}")

    print(f"\nPer-fold details:")
    for row in nested.get("fold_rows", []):
        tm = row.get("test_metrics", {})
        print(f"  Fold {row['fold_id']}: test [{row['test_start']} → {row['test_end']}] "
              f"Ret={tm.get('return_pct', 0):+.2f}% DD={tm.get('drawdown_pct', 0):.2f}% "
              f"PF={tm.get('profit_factor', 0):.3f} Trades={tm.get('trades', 0)}")
else:
    print("No nested_walk_forward.json found.")

## 5. Phase-by-Phase Comparison

In [ ]:
# Compare best candidates from each optimization phase
if RESULT_DIR:
    phase_files = {
        "Discrete": "phase_discrete.csv",
        "Coarse LHS": "phase_coarse_lhs.csv",
        "TPE": "phase_tpe.csv",
        "NSGA-II": "phase_nsga2.csv",
    }
    summary_rows = []
    for label, fname in phase_files.items():
        fpath = RESULT_DIR / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            if not df.empty:
                best = df.iloc[0]  # Already sorted by objective_score desc
                summary_rows.append({
                    "Phase": label,
                    "Trials": len(df),
                    "Best Obj": f"{best.get('objective_score', 0):.4f}",
                    "OOS Ret%": f"{best.get('oos_return_pct', 0):.2f}",
                    "OOS DD%": f"{best.get('oos_drawdown_pct', 0):.2f}",
                    "OOS PF": f"{best.get('oos_pf', 0):.3f}",
                    "OOS Trades": int(best.get('oos_trades', 0)),
                    "Overfit": f"{best.get('overfit_gap', 0):.4f}",
                })

    if summary_rows:
        print("Phase Comparison (best candidate per phase):")
        display(pd.DataFrame(summary_rows))
    else:
        print("No phase CSVs found.")

## 6. Copy Results to Google Drive (Optional)

In [ ]:
# Uncomment to copy results to your Drive
# import shutil
# DRIVE_DEST = "/content/drive/MyDrive/Python-Algo/optimization_results"
# if RESULT_DIR:
#     dest = Path(DRIVE_DEST) / RESULT_DIR.name
#     shutil.copytree(RESULT_DIR, dest, dirs_exist_ok=True)
#     print(f"Results copied to {dest}")

In [ ]:
## 7. Download Results to Local Machine

# Create zip file with all results and download
import shutil
import subprocess
from pathlib import Path

if RESULT_DIR:
    # Create a zip file
    zip_filename = f"v6_2_optimization_{RESULT_DIR.name}"
    zip_path = Path(f"/tmp/{zip_filename}.zip")
    
    print(f"Creating zip archive...")
    shutil.make_archive(str(zip_path).replace('.zip', ''), 'zip', RESULT_DIR.parent, RESULT_DIR.name)
    print(f"✓ Zip created: {zip_path}")
    print(f"✓ Size: {zip_path.stat().st_size / (1024*1024):.1f} MB")
    
    # Download
    from google.colab import files
    print(f"\nDownloading {zip_filename}.zip...")
    files.download(str(zip_path))
    print("✓ Download complete!")
    
    # Also print the key files summary
    print(f"\n📊 Results Summary:")
    print(f"  ├─ deployment_ranking.csv (sortable by eligible_for_deploy)")
    print(f"  ├─ best_candidate_report.json (top candidate params & stats)")
    print(f"  ├─ nested_walk_forward.json (per-fold performance)")
    print(f"  ├─ phase_discrete.csv, phase_coarse_lhs.csv, phase_tpe.csv, phase_nsga2.csv")
    print(f"  ├─ shortlist.csv (top 40 candidates)")
    print(f"  ├─ baseline_fold_test.csv (V6.2 default performance)")
    print(f"  └─ run_config.json (full execution metadata)")
else:
    print("No results directory found. Run optimization first.")